FUNÇÕES DE MÉTRICA: P@K, AP E NDCG:

In [6]:
precisao_em_k <- function(ranking, graus, k) {
  topo <- names(ranking)[1:k]
  relevantes <- sum(graus[topo] > 0)
  relevantes / k
}

average_precision <- function(ranking, graus) {
  nomes <- names(ranking)
  relevante <- graus[nomes] > 0
  if (sum(relevante) == 0) return(0)
  precisoes <- sapply(seq_along(nomes), function(i) {
    if (!relevante[i]) return(NA)
    sum(relevante[1:i]) / i
  })
  mean(precisoes, na.rm = TRUE)
}

dcg_em_k <- function(graus_ordenados, k) {
  g <- graus_ordenados[1:k]
  sum(g / log2(seq_len(k) + 1))
}

ndcg_em_k <- function(ranking, graus, k) {
  graus_no_ranking <- graus[names(ranking)]
  dcg <- dcg_em_k(graus_no_ranking, k)
  graus_ideais <- sort(graus, decreasing = TRUE)
  idcg <- dcg_em_k(graus_ideais, k)
  if (idcg == 0) return(0)
  dcg / idcg
}

Recriando o corpus e os rankings das Aulas 02 e 04 (igual já está no seu notebook):

In [1]:
docs <- c(
  d1 = "recuperacao de informacao ordena documentos por relevancia",
  d2 = "o modelo de espaco vetorial representa documentos como vetores",
  d3 = "bm25 e um modelo probabilistico de ranqueamento de texto",
  d4 = "aprendizado estatistico fundamenta a recuperacao moderna",
  d5 = "o indice invertido acelera a busca em muitos documentos",
  d6 = "embeddings capturam a semantica de palavras e documentos",
  d7 = "a avaliacao mede a relevancia dos resultados da busca",
  d8 = "ciencia de dados combina estatistica e programacao"
)

tok <- function(x) unlist(strsplit(tolower(x), "\\s+"))
tokens <- lapply(docs, tok)
vocab <- sort(unique(unlist(tokens)))

tf <- sapply(tokens, function(t) as.integer(table(factor(t, levels = vocab))))
rownames(tf) <- vocab
dl <- colSums(tf)
avgdl <- mean(dl)

N <- ncol(tf)
df <- rowSums(tf > 0)
idf_classico <- log(N / df)
idf_bm25 <- log(N - df + 0.5) / (df + 0.5 + 1)

Funções de busca, iguais às que vocês já usaram:

In [2]:
cosseno <- function(a, b) sum(a * b) / (sqrt(sum(a^2)) * sqrt(sum(b^2)))

buscar_tfidf <- function(consulta) {
  termos <- tok(consulta)
  q <- as.integer(table(factor(termos, levels = vocab)))
  qw <- q * idf_classico
  w <- tf * idf_classico
  scores <- apply(w, 2, function(dvec) cosseno(qw, dvec))
  sort(scores, decreasing = TRUE)
}

bm25_doc <- function(termos, d, k1, b) {
  s <- 0
  for (t in termos) {
    if (!(t %in% vocab)) next
    f <- tf[t, d]
    if (f == 0) next
    K <- k1 * (1 - b + b * dl[d] / avgdl)
    s <- s + unname(idf_bm25[t]) * (f * (k1 + 1)) / (f + K)
  }
  unname(s)
}

buscar_bm25 <- function(consulta, k1 = 1.2, b = 0.75) {
  termos <- tok(consulta)
  scores <- sapply(colnames(tf), function(d) bm25_doc(termos, d, k1, b))
  sort(scores, decreasing = TRUE)
}

consultas <- c(
  "modelo de recuperacao",
  "busca e relevancia dos documentos",
  "estatistica e ciencia de dados"
)

Funções de busca, iguais às que vocês já usaram:

In [ ]:
cosseno <- function(a, b) sum(a * b) / (sqrt(sum(a^2)) * sqrt(sum(b^2)))

buscar_tfidf <- function(consulta) {
  termos <- tok(consulta)
  q <- as.integer(table(factor(termos, levels = vocab)))
  qw <- q * idf_classico
  w <- tf * idf_classico
  scores <- apply(w, 2, function(dvec) cosseno(qw, dvec))
  sort(scores, decreasing = TRUE)
}

bm25_doc <- function(termos, d, k1, b) {
  s <- 0
  for (t in termos) {
    if (!(t %in% vocab)) next
    f <- tf[t, d]
    if (f == 0) next
    K <- k1 * (1 - b + b * dl[d] / avgdl)
    s <- s + unname(idf_bm25[t]) * (f * (k1 + 1)) / (f + K)
  }
  unname(s)
}

buscar_bm25 <- function(consulta, k1 = 1.2, b = 0.75) {
  termos <- tok(consulta)
  scores <- sapply(colnames(tf), function(d) bm25_doc(termos, d, k1, b))
  sort(scores, decreasing = TRUE)
}

consultas <- c(
  "modelo de recuperacao",
  "busca e relevancia dos documentos",
  "estatistica e ciencia de dados"
)

Gabarito de relevância pras 3 consultas (0 não relevante, 1 parcial, 2 relevante), descrito acima:

In [3]:
gabarito <- list(
  "modelo de recuperacao" = c(
    d1 = 2, d2 = 2, d3 = 2, d4 = 1, d5 = 0, d6 = 1, d7 = 0, d8 = 0
  ),
  "busca e relevancia dos documentos" = c(
    d1 = 2, d2 = 0, d3 = 1, d4 = 0, d5 = 2, d6 = 0, d7 = 2, d8 = 0
  ),
  "estatistica e ciencia de dados" = c(
    d1 = 0, d2 = 0, d3 = 0, d4 = 2, d5 = 0, d6 = 0, d7 = 0, d8 = 2
  )
)

Funções de P@3, AP (que vira MAP quando tirada a média entre consultas) e nDCG@3:

In [ ]:
precisao_em_k <- function(ranking, graus, k) {
  topo <- names(ranking)[1:k]
  relevantes <- sum(graus[topo] > 0)
  relevantes / k
}

average_precision <- function(ranking, graus) {
  nomes <- names(ranking)
  relevante <- graus[nomes] > 0
  if (sum(relevante) == 0) return(0)
  precisoes <- sapply(seq_along(nomes), function(i) {
    if (!relevante[i]) return(NA)
    sum(relevante[1:i]) / i
  })
  mean(precisoes, na.rm = TRUE)
}

dcg_em_k <- function(graus_ordenados, k) {
  g <- graus_ordenados[1:k]
  sum(g / log2(seq_len(k) + 1))
}

ndcg_em_k <- function(ranking, graus, k) {
  graus_no_ranking <- graus[names(ranking)]
  dcg <- dcg_em_k(graus_no_ranking, k)
  graus_ideais <- sort(graus, decreasing = TRUE)
  idcg <- dcg_em_k(graus_ideais, k)
  if (idcg == 0) return(0)
  dcg / idcg
}

Rodando os dois sistemas nas 3 consultas e juntando tudo numa tabela:

In [7]:
resultados <- data.frame()

for (cons in consultas) {
  graus <- gabarito[[cons]]
  rank_tfidf <- buscar_tfidf(cons)
  rank_bm25 <- buscar_bm25(cons)

  resultados <- rbind(resultados,
    data.frame(consulta = cons, sistema = "TF-IDF",
      p_em_3 = precisao_em_k(rank_tfidf, graus, 3),
      ap = average_precision(rank_tfidf, graus),
      ndcg_em_3 = ndcg_em_k(rank_tfidf, graus, 3)),
    data.frame(consulta = cons, sistema = "BM25",
      p_em_3 = precisao_em_k(rank_bm25, graus, 3),
      ap = average_precision(rank_bm25, graus),
      ndcg_em_3 = ndcg_em_k(rank_bm25, graus, 3))
  )
}

print(resultados, row.names = FALSE, digits = 3)

medias <- aggregate(cbind(p_em_3, ap, ndcg_em_3) ~ sistema, data = resultados, FUN = mean)
names(medias)[names(medias) == "ap"] <- "map"
print(medias, digits = 3)

                          consulta sistema p_em_3    ap ndcg_em_3
             modelo de recuperacao  TF-IDF  1.000 1.000     0.883
             modelo de recuperacao    BM25  1.000 0.967     1.000
 busca e relevancia dos documentos  TF-IDF  1.000 0.917     1.000
 busca e relevancia dos documentos    BM25  1.000 0.917     1.000
    estatistica e ciencia de dados  TF-IDF  0.333 0.667     0.613
    estatistica e ciencia de dados    BM25  0.333 0.667     0.613
  sistema p_em_3   map ndcg_em_3
1    BM25  0.778 0.850     0.871
2  TF-IDF  0.778 0.861     0.832
